# Mortgage default preprocessing

This notebook creates the training and validation inputs and fits a reusable preprocessing pipeline. All learned preprocessing parameters come from the 2008–2015 training period. The 2017–2018 test set, missing-date records, and unlabelled records remain untouched.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

## 1. Load the source table

The database query is read-only. Credentials come from the Git-ignored `.env` file and are never displayed.

In [2]:
repo_root = Path.cwd()
if not (repo_root / ".env").exists():
    repo_root = repo_root.parent

env_path = repo_root / ".env"
if not env_path.exists():
    raise FileNotFoundError("Could not find the repository .env file.")
load_dotenv(env_path)

required_settings = ["PGHOST", "PGPORT", "PGDATABASE", "PGUSER", "PGPASSWORD"]
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise ValueError(f"Missing database settings: {', '.join(missing_settings)}")

database_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],
    host=os.environ["PGHOST"],
    port=int(os.environ["PGPORT"]),
    database=os.environ["PGDATABASE"],
)
query = """
SELECT
    source_row_id, income, loan_amount, term_length, install_to_inc,
    occup, marital, schufa, num_applic, obs_date, target_var
FROM public.mortgage_default
ORDER BY source_row_id
"""
engine = create_engine(database_url)
try:
    mortgage_data = pd.read_sql_query(query, engine)
finally:
    engine.dispose()

mortgage_data["obs_date"] = pd.to_datetime(mortgage_data["obs_date"], errors="raise")
mortgage_data["target_var"] = mortgage_data["target_var"].astype("Int64")
assert len(mortgage_data) == 10_000
assert mortgage_data["source_row_id"].is_unique
print(f"Loaded {len(mortgage_data):,} records.")

Loaded 10,000 records.


## 2. Recreate the frozen split

Training covers 2008–2015 and validation covers 2016. Test records from 2017–2018 are identified only so leakage assertions can verify that they do not enter preprocessing.

In [3]:
labelled_data = mortgage_data.loc[mortgage_data["target_var"].notna()].copy()
observation_year = labelled_data["obs_date"].dt.year

training_data = labelled_data.loc[observation_year.le(2015)].copy()
validation_data = labelled_data.loc[observation_year.eq(2016)].copy()
test_ids = set(labelled_data.loc[observation_year.isin([2017, 2018]), "source_row_id"])
missing_date_ids = set(labelled_data.loc[labelled_data["obs_date"].isna(), "source_row_id"])
unlabelled_ids = set(mortgage_data.loc[mortgage_data["target_var"].isna(), "source_row_id"])

training_ids = set(training_data["source_row_id"])
validation_ids = set(validation_data["source_row_id"])
assert len(training_data) == 6_422
assert len(validation_data) == 838
assert training_ids.isdisjoint(validation_ids | test_ids | missing_date_ids | unlabelled_ids)
assert validation_ids.isdisjoint(test_ids | missing_date_ids | unlabelled_ids)

pd.DataFrame(
    {
        "partition": ["Training", "Validation"],
        "records": [len(training_data), len(validation_data)],
        "defaults": [training_data["target_var"].sum(), validation_data["target_var"].sum()],
        "default_rate_pct": [
            100 * training_data["target_var"].mean(),
            100 * validation_data["target_var"].mean(),
        ],
    }
).round(2)

,partition,records,defaults,default_rate_pct
0,Training,6422,973,15.15
1,Validation,838,146,17.42


## 3. Define model inputs and target

The initial model uses five continuous variables plus occupation and applicant count. Marital status is excluded under the documented primary-model strategy. Observation date and source row ID support validation and traceability but are not predictors.

In [4]:
NUMERIC_FEATURES = [
    "income", "loan_amount", "term_length",
    "install_to_inc", "schufa",
]
CATEGORICAL_FEATURES = ["occupation", "number_of_applicants"]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

def prepare_model_features(data):
    """Apply deterministic naming and type conversions without learning from data."""
    features = data[NUMERIC_FEATURES].copy()

    occupation = data["occup"].astype("string")
    features["occupation"] = pd.Series(
        occupation.to_numpy(dtype=object, na_value=np.nan), index=data.index
    )

    applicant_count = data["num_applic"].astype("Int64").astype("string")
    features["number_of_applicants"] = pd.Series(
        applicant_count.to_numpy(dtype=object, na_value=np.nan), index=data.index
    )
    return features[MODEL_FEATURES]

X_train = prepare_model_features(training_data)
y_train = training_data["target_var"].astype(int).copy()
X_validation = prepare_model_features(validation_data)
y_validation = validation_data["target_var"].astype(int).copy()

assert X_train.index.equals(y_train.index)
assert X_validation.index.equals(y_validation.index)
assert list(X_train.columns) == MODEL_FEATURES
assert not ({"source_row_id", "marital", "obs_date", "target_var"} & set(X_train.columns))

pd.DataFrame(
    {
        "dataset": ["X_train", "X_validation"],
        "rows": [len(X_train), len(X_validation)],
        "input_features": [X_train.shape[1], X_validation.shape[1]],
    }
)

,dataset,rows,input_features
0,X_train,6422,7
1,X_validation,838,7


## 4. Fit preprocessing on training data only

Continuous values receive median imputation and standard scaling. Separate missing indicators preserve whether each value was originally absent. Categorical values receive an explicit `Unknown` category and one-hot encoding. Unseen validation categories are ignored safely.

In [5]:
numeric_value_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False,
            drop=["Employee", "1"],
        )),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_value_pipeline, NUMERIC_FEATURES),
        ("numeric_missing", MissingIndicator(features="all"), NUMERIC_FEATURES),
        ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
)
preprocessor.set_output(transform="pandas")

# This is the only fitting operation: validation data is never passed to fit.
X_train_processed = preprocessor.fit_transform(X_train)
X_validation_processed = preprocessor.transform(X_validation)

print(f"Training matrix: {X_train_processed.shape}")
print(f"Validation matrix: {X_validation_processed.shape}")

Training matrix: (6422, 18)
Validation matrix: (838, 18)


## 5. Verify the transformed data

These assertions stop the notebook if preprocessing changes row counts, misaligns the two matrices, leaves invalid values, or exposes excluded fields.

In [6]:
assert len(X_train_processed) == len(X_train) == len(y_train)
assert len(X_validation_processed) == len(X_validation) == len(y_validation)
assert list(X_train_processed.columns) == list(X_validation_processed.columns)
assert not X_train_processed.isna().any().any()
assert not X_validation_processed.isna().any().any()
assert np.isfinite(X_train_processed.to_numpy(dtype=float)).all()
assert np.isfinite(X_validation_processed.to_numpy(dtype=float)).all()
assert not any(
    excluded in column
    for excluded in ["source_row_id", "marital", "obs_date", "target_var"]
    for column in X_train_processed.columns
)

feature_audit = pd.DataFrame(
    {
        "transformed_feature": X_train_processed.columns,
        "training_missing_values": X_train_processed.isna().sum().to_numpy(),
        "validation_missing_values": X_validation_processed.isna().sum().to_numpy(),
    }
)
feature_audit

,transformed_feature,training_missing_values,validation_missing_values
0,numeric__income,0,0
1,numeric__loan_amount,0,0
2,numeric__term_length,0,0
3,numeric__install_to_inc,0,0
4,numeric__schufa,0,0
5,numeric_missing__missingindicator_income,0,0
6,numeric_missing__missingindicator_loan_amount,0,0
7,numeric_missing__missingindicator_term_length,0,0
8,numeric_missing__missingindicator_install_to_inc,0,0
9,numeric_missing__missingindicator_schufa,0,0


## Preprocessing conclusion

The preprocessing pipeline is now ready to attach to a logistic-regression model. It learns medians, scaling parameters, and category levels from training data only, then applies the same transformation to validation data. The next step is to keep this preprocessor and model together in one scikit-learn pipeline so cross-validation, probability prediction, and later deployment cannot accidentally apply inconsistent transformations.